# SSTW S1 real DiT relation primitive

> **运行提示：L4 可用于本次 S1 稀疏 native-SDPA 诊断。Notebook 只检查 CUDA 与 BF16 能力，并打印实际 GPU 供诊断。**

METHOD_ONLY / DIAGNOSTIC_ONLY. Runs one frozen exact20 internal-stat diagnostic. It does not run VAE, encode MP4, or enter S2-S4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
AUTHORIZED_REF = '7a083b9907dfdc68c7c972fc469fb522b9667b2e'
RUN_ID = '025f2346a837106c'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/SC-SSTW-Feasibility/s1-block-probe-batch-shape'
AUTHORIZE_EXECUTION = True
AUTHORIZE_DRIVE_IO = True
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
MODEL_REVISION = '0fad780a534b6463e45facd96134c9f345acfa5b'
EXPECTED_CONFIG_SHA256 = '0dad730be27ef10df9db649e7a6bb57d329ed3be5c7f0aa18677389ad2b5d212'


In [ ]:
import subprocess
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False, capture_output=True, text=True)
print('GPU diagnostic:', gpu_probe.stdout.strip() or gpu_probe.stderr.strip() or 'unavailable')
print('Runtime diagnostic only; L4 is supported for this sparse native-SDPA S1 run.')


In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys, zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_absent(*paths):
    for path in paths:
        if path.exists() or path.is_symlink():
            raise RuntimeError(f'refusing to overwrite: {path}')

if not AUTHORIZE_EXECUTION or not AUTHORIZE_DRIVE_IO:
    raise RuntimeError('explicit execution and Drive acknowledgements are required')
if not re.fullmatch(r'[0-9a-f]{40}', AUTHORIZED_REF) or not re.fullmatch(r'[0-9a-f]{16}', RUN_ID):
    raise RuntimeError('exact ref and frozen run id are required')
WORK = Path('/content') / f'sstw-s1-source-{RUN_ID}'
LOCAL_ROOT = Path('/content') / f'sstw-s1-run-{RUN_ID}'
OUTPUT = LOCAL_ROOT / 'output'
LOG = Path('/content') / f'sstw-s1-log-{RUN_ID}'
BUNDLE = Path('/content') / f'sstw-s1-bundle-{RUN_ID}'
ARCHIVE = Path('/content') / f'sstw-s1-block-probe-{RUN_ID}.zip'
SIDECAR = Path('/content') / f'sstw-s1-block-probe-{RUN_ID}.zip.sha256.json'
DRIVE_ROOT = Path(DRIVE_OUTPUT_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT.is_symlink() or not DRIVE_ROOT.is_dir():
    raise RuntimeError('Drive output root is not a regular directory')
DRIVE_ARCHIVE = DRIVE_ROOT / ARCHIVE.name
DRIVE_SIDECAR = DRIVE_ROOT / SIDECAR.name
require_absent(WORK, LOCAL_ROOT, OUTPUT, LOG, BUNDLE, ARCHIVE, SIDECAR, DRIVE_ARCHIVE, DRIVE_SIDECAR)
LOG.mkdir()
runner_started = False
completed = None
audit = None
caught = None
try:
    with (LOG / 'clone.stdout').open('xb') as out, (LOG / 'clone.stderr').open('xb') as err:
        subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(WORK)], check=True, stdout=out, stderr=err)
    with (LOG / 'checkout.stdout').open('xb') as out, (LOG / 'checkout.stderr').open('xb') as err:
        subprocess.run(['git', 'checkout', '--detach', AUTHORIZED_REF], cwd=WORK, check=True, stdout=out, stderr=err)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(['git', 'status', '--porcelain=v1', '--untracked-files=all'], cwd=WORK, check=True, capture_output=True, text=True).stdout
    if head != AUTHORIZED_REF or dirty:
        raise RuntimeError('clean exact-ref checkout failed')
    config_path = WORK / 'configs/s1_real_dit_relation_primitive.json'
    if sha256_file(config_path) != EXPECTED_CONFIG_SHA256:
        raise RuntimeError('frozen S1 config identity mismatch')
    expected_run_id = hashlib.sha256(('SSTW-S1-BLOCK-PROBE-BATCH-SHAPE:' + AUTHORIZED_REF + ':' + EXPECTED_CONFIG_SHA256).encode()).hexdigest()[:16]
    if RUN_ID != expected_run_id:
        raise RuntimeError('run id does not bind exact ref and config')
    locked = ['accelerate==1.4.0', 'diffusers==0.35.2', 'ftfy==6.3.1', 'huggingface_hub==0.35.3', 'numpy==1.26.4', 'safetensors==0.5.3', 'transformers==4.49.0']
    with (LOG / 'pip.stdout').open('xb') as out, (LOG / 'pip.stderr').open('xb') as err:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *locked], check=True, stdout=out, stderr=err)
    import torch
    from huggingface_hub import snapshot_download
    cuda_available = bool(torch.cuda.is_available())
    bf16_supported = bool(cuda_available and torch.cuda.is_bf16_supported())
    gpu_name = None if not cuda_available else torch.cuda.get_device_name(0)
    print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name, 'cuda_available': cuda_available, 'bf16_supported': bf16_supported}, sort_keys=True))
    if not cuda_available or not bf16_supported:
        raise RuntimeError('S1 requires CUDA and BF16 capabilities')
    snapshot = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, local_files_only=False))
    resolved_snapshot = snapshot.resolve(strict=True)
    if snapshot.is_symlink() or not snapshot.is_dir() or resolved_snapshot.name != MODEL_REVISION:
        raise RuntimeError('downloaded model snapshot identity mismatch')
    runtime = {'head': head, 'run_id': RUN_ID, 'config_sha256': EXPECTED_CONFIG_SHA256, 'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name, 'cuda_available': cuda_available, 'bf16_supported': bf16_supported, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'snapshot': str(resolved_snapshot), 'diagnostic_class': 'DIAGNOSTIC_ONLY'}
    (LOG / 'runtime.json').write_text(json.dumps(runtime, sort_keys=True, indent=2) + '\n')
    LOCAL_ROOT.mkdir()
    argv = [sys.executable, str(WORK / 'experiments/run_s1_real_dit_relation_primitive.py'), '--output', str(OUTPUT)]
    runner_started = True
    with (LOG / 'runner.stdout').open('xb') as out, (LOG / 'runner.stderr').open('xb') as err:
        completed = subprocess.run(argv, cwd=WORK, stdout=out, stderr=err)
    runner_stdout = (LOG / 'runner.stdout').read_text(encoding='utf-8', errors='replace')
    if runner_stdout.strip():
        print('S1 runner stdout:')
        print(runner_stdout.rstrip())
    if OUTPUT.is_symlink() or not (OUTPUT / 'audit.json').is_file():
        runner_report = None
        for line in reversed([line for line in runner_stdout.splitlines() if line.strip()]):
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict):
                runner_report = candidate
                break
        if runner_report is not None:
            raise RuntimeError('runner did not publish audit: ' + json.dumps(runner_report, sort_keys=True))
        raise RuntimeError('runner audit missing; runner stdout was displayed above')
    audit = json.loads((OUTPUT / 'audit.json').read_text())
    expected_codes = {'S1_GO': 0, 'S1_NO_GO_THIS_CONSTRUCTION': 3}
    if audit.get('status') not in expected_codes or completed.returncode != expected_codes[audit['status']]:
        raise RuntimeError('runner exit/status mismatch')
    if audit.get('transformer_calls') != 20 or audit.get('scheduler_steps') != 4 or audit.get('VAE_was_run') is not False or audit.get('MP4_was_written') is not False:
        raise RuntimeError('S1 execution budget or boundary changed')
except BaseException as exc:
    caught = exc
finally:
    BUNDLE.mkdir()
    shutil.copytree(LOG, BUNDLE / 'log')
    if OUTPUT.exists() and OUTPUT.is_dir() and not OUTPUT.is_symlink():
        shutil.copytree(OUTPUT, BUNDLE / 'output')
    if WORK.exists() and WORK.is_dir() and not WORK.is_symlink():
        frozen = BUNDLE / 'frozen'
        frozen.mkdir()
        for relative in ('SSTW_METHOD_AUTHORITY.md', 'configs/s1_real_dit_relation_primitive.json', 'plans/s1_real_dit_relation_primitive.json'):
            source = WORK / relative
            if source.is_file() and not source.is_symlink():
                shutil.copy2(source, frozen / Path(relative).name)
    state = {'schema': 'sstw.s1.notebook_state.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'authorized_ref': AUTHORIZED_REF, 'runner_started': runner_started, 'runner_return_code': None if completed is None else completed.returncode, 'audit_status': None if audit is None else audit.get('status'), 'failure_type': None if caught is None else type(caught).__name__, 'formal_result': False, 'stage_progression_allowed': False}
    (BUNDLE / 'notebook_state.json').write_text(json.dumps(state, sort_keys=True, indent=2) + '\n')
    shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', BUNDLE)
    with zipfile.ZipFile(ARCHIVE) as handle:
        if handle.testzip() is not None:
            raise RuntimeError('local ZIP integrity failed')
    archive_sha = sha256_file(ARCHIVE)
    sidecar = {'schema': 'sstw.s1.archive_identity.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'source_ref': AUTHORIZED_REF, 'archive_name': ARCHIVE.name, 'archive_size': ARCHIVE.stat().st_size, 'archive_sha256': archive_sha, 'formal_result': False, 'stage_progression_allowed': False}
    SIDECAR.write_text(json.dumps(sidecar, sort_keys=True, indent=2) + '\n')
    with ARCHIVE.open('rb') as source, DRIVE_ARCHIVE.open('xb') as target:
        shutil.copyfileobj(source, target)
    with SIDECAR.open('rb') as source, DRIVE_SIDECAR.open('xb') as target:
        shutil.copyfileobj(source, target)
    if DRIVE_ARCHIVE.stat().st_size != ARCHIVE.stat().st_size or sha256_file(DRIVE_ARCHIVE) != archive_sha:
        raise RuntimeError('Drive ZIP readback mismatch')
    if DRIVE_SIDECAR.read_bytes() != SIDECAR.read_bytes():
        raise RuntimeError('Drive sidecar readback mismatch')
    with zipfile.ZipFile(DRIVE_ARCHIVE) as handle:
        if handle.testzip() is not None:
            raise RuntimeError('Drive ZIP integrity failed')
if caught is not None:
    raise caught
print({'status': audit['status'], 'run_id': RUN_ID, 'drive_zip': str(DRIVE_ARCHIVE), 'drive_sidecar': str(DRIVE_SIDECAR)})
